## Ingeniería de Características: Creación de Variables Derivadas

### 1. Justificación Clínica y Operativa
Para maximizar la capacidad predictiva de los algoritmos de Machine Learning sobre el costo y la severidad de los episodios GRD, se construyeron variables derivadas que capturan la "intensidad" del caso clínico mejor que los datos tabulares crudos.

#### A. Variables de Carga de Enfermedad (Comorbilidades)
* **`NUM_COMORBILIDADES`:** Conteo de diagnósticos secundarios válidos (excluyendo el cáncer base y códigos administrativos). Un paciente con múltiples comorbilidades (ej. diabetes + EPOC + falla renal) tiene un riesgo inherentemente mayor de complicaciones postoperatorias y estadías prolongadas.
* **`CARGA_ONCOLOGICA`:** Conteo estricto de códigos neoplásicos (C00-C97, D00-D48). Diferencia a un paciente con un tumor primario único de uno con enfermedad metastásica multisistémica.
* **`COMORBILIDAD_PRINCIPAL`:** Mapeo de la primera comorbilidad válida a los 22 Macrogrupos del CIE-10 (ej. *Endocrinas y Metabólicas*). Para el grupo control (no oncológico), se asigna la etiqueta "NO_APLICA", ya que su diagnóstico principal ya define su condición base.

#### B. Variables de Intensidad Terapéutica
* **`NUM_PROCEDIMIENTOS`:** Sumatoria de intervenciones realizadas (máximo 30). Altamente correlacionado con el uso de insumos, horas de pabellón y complejidad de enfermería.
* **`ES_QUIRURGICO`:** Variable binaria (1/0) que clasifica el episodio identificando si el `PROCEDIMIENTO1` corresponde a una intervención con incisión/pabellón (capítulos 01 al 86 del CIE-9-MC) o si fue un ingreso puramente médico, terapéutico o de soporte vital.
* **`CANTIDAD_TRASLADOS`:** Métrica de inestabilidad clínica. Cuantifica los movimientos del paciente entre distintas unidades del hospital (ej. Urgencia -> Pabellón -> UCI -> Sala Básica).

#### C. Variables Temporales y Demográficas
* **`EDAD`:** Calculada en años cronológicos al momento del ingreso. Es el predictor demográfico primario de riesgo de mortalidad.
* **`DIAS_ESTADIA`:** Diferencia en días entre la fecha de alta y la fecha de ingreso. Representa el consumo principal del recurso "día-cama".

### 2. Análisis de Calidad y Manejo de Nulos (Anomalías Temporales)
Al calcular las variables numéricas, las columnas de conteo (`NUM_COMORBILIDADES`, `CANTIDAD_TRASLADOS`, etc.) presentaron un **0% de valores nulos**. 

Sin embargo, el cruce de fechas para calcular `EDAD` y `DIAS_ESTADIA` reveló inconsistencias de digitación administrativa en el origen de los datos MINSAL (ej. fechas de alta anteriores a la fecha de ingreso, o ingresos maternos heredados a recién nacidos generando edades negativas).
* **Filtro de Corrección:** Se aplicó un algoritmo de parseo estricto adaptado a la mutación de formatos del MINSAL (paso de `yyyy-mm-dd` a `dd-mm-yyyy` en 2023), rescatando más de 1 millón de registros anuales.
* **Tratamiento de Errores Residuales:** Las anomalías negativas reales se transformaron a nulos (`NaN`). La auditoría final arrojó un volumen estadísticamente insignificante de errores irrecuperables (Máximo histórico: 53 casos nulos en `EDAD` y 50 en `DIAS_ESTADIA` durante 2024, sobre un total de >1.076.000 egresos).
* **Veredicto:** **Eliminación Directa**. Dado que estos valores nulos representan consistentemente menos del **0.005%** de la cohorte anual, se procede a la eliminación de las filas (Listwise Deletion). La imputación estadística no se justifica y podría introducir sesgos al crear tiempos de estadía sintéticos en episodios con fechas corruptas.

In [1]:
import pandas as pd
import numpy as np
import os
import re

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_procesados = [f"GRD_PROCESADO_{año}.csv" for año in range(2019, 2025)]

print("Iniciando creación de variables derivadas...\n" + "="*60)

# 2. Configuración para Comorbilidades
letras_excluidas = ['Z', 'V', 'W', 'X', 'Y']

condiciones_macro_letras = [
    ['A', 'B'], ['D'], ['E'], ['F'],
    ['G'], ['H'], ['I'], ['J'],
    ['K'], ['L'], ['M'], ['N'],
    ['O', 'P', 'Q'], ['R'], ['S', 'T'], ['U']
]
opciones_macro = [
    "INFECCIOSAS Y PARASITARIAS", "SANGRE E INMUNIDAD", "ENDOCRINAS Y METABOLICAS", "TRASTORNOS MENTALES",
    "SISTEMA NERVIOSO", "OJO Y OIDO", "SISTEMA CIRCULATORIO", "SISTEMA RESPIRATORIO",
    "SISTEMA DIGESTIVO", "PIEL Y TEJIDO SUBCUTANEO", "SISTEMA MUSCULOESQUELETICO", "SISTEMA GENITOURINARIO",
    "MATERNO-INFANTILES Y CONGENITAS", "SINTOMAS Y HALLAZGOS", "TRAUMATISMOS Y ENVENENAMIENTOS", "CODIGOS PROVISIONALES (COVID)"
]

def obtener_macro_comorbilidad(letra):
    for letras_cond, macro in zip(condiciones_macro_letras, opciones_macro):
        if letra in letras_cond:
            return macro
    return np.nan

for archivo in archivos_procesados:
    ruta_origen = os.path.join(carpeta_procesados, archivo)
    año = archivo[-8:-4]
    
    df = pd.read_csv(ruta_origen, low_memory=False)
    
    # ---------------------------------------------------------
    # A. CANTIDAD DE TRASLADOS INTERNOS
    # ---------------------------------------------------------
    cols_traslados = [f"SERVICIOTRASLADO{i}" for i in range(1, 10)]
    df['CANTIDAD_TRASLADOS'] = df[cols_traslados].notnull().sum(axis=1)
    
    # ---------------------------------------------------------
    # B. NÚMERO DE PROCEDIMIENTOS Y BANDERA QUIRÚRGICA
    # ---------------------------------------------------------
    cols_procedimientos = [f"PROCEDIMIENTO{i}" for i in range(1, 31)]
    df['NUM_PROCEDIMIENTOS'] = df[cols_procedimientos].notnull().sum(axis=1)
    
    # Determinar si es Quirúrgico usando el código CIE-9 original (antes de la transformación a texto)
    # Los capítulos 01 al 86 son Quirúrgicos. 87 a 99 (y 00, 17) son Diagnósticos/Terapéuticos.
    def es_quirurgico(proc):
        proc_str = str(proc).strip().upper()
        match = re.match(r'^0*(\d{1,2})', proc_str)
        if match:
            macro = int(match.group(1))
            # Rango quirúrgico estándar (01 a 86). Excluimos partos naturales menores si se desea, 
            # pero estructuralmente todo lo que implica pabellón/incisión está aquí.
            if 1 <= macro <= 86 and macro != 17: 
                return 1
        return 0

    df['ES_QUIRURGICO'] = df['PROCEDIMIENTO1'].apply(es_quirurgico)
    
    # ---------------------------------------------------------
    # C. CARGA ONCOLÓGICA, NUM_COMORBILIDADES Y COMORB. PRINCIPAL
    # ---------------------------------------------------------
    cols_diagnosticos = [f"DIAGNOSTICO{i}" for i in range(1, 36)]
    
    def procesar_diagnosticos(fila):
        cancer = 0
        comorbilidad = 0
        comorbilidad_principal = np.nan
        es_oncologico = fila.get('CATEGORIA_CANCER', 'SIN_CANCER') != 'SIN_CANCER'
        
        # Iterar sobre las 35 columnas
        for col in cols_diagnosticos:
            val = fila[col]
            if pd.isna(val):
                continue
                
            val_str = str(val).strip().upper()
            if val_str in ["DESCONOCIDO", "S/D", ""]:
                continue
                
            extraccion = re.match(r'^([A-Z])(\d{2})', val_str)
            if not extraccion:
                continue
                
            letra = extraccion.group(1)
            numero = float(extraccion.group(2))
            
            # Clasificación CIE-10
            es_codigo_cancer = (letra == 'C') or (letra == 'D' and numero <= 48)
            es_codigo_admin = letra in letras_excluidas
            
            if es_codigo_cancer:
                cancer += 1
            elif not es_codigo_admin:
                comorbilidad += 1
                # Capturar la primera comorbilidad válida si es oncológico
                if es_oncologico and pd.isna(comorbilidad_principal):
                    comorbilidad_principal = obtener_macro_comorbilidad(letra)
        
        # Asignaciones finales según el tipo de paciente
        if not es_oncologico:
            comorbilidad_principal = "NO_APLICA"
        elif pd.isna(comorbilidad_principal):
            comorbilidad_principal = "SIN_COMORBILIDAD"
            
        return pd.Series([cancer, comorbilidad, comorbilidad_principal])

    df[['CARGA_ONCOLOGICA', 'NUM_COMORBILIDADES', 'COMORBILIDAD_PRINCIPAL']] = df.apply(procesar_diagnosticos, axis=1)
    
    # ---------------------------------------------------------
    # D. MANEJO DE FECHAS (EDAD y DIAS_ESTADIA) - PARCHE BASADO EN AUDITORÍA
    # ---------------------------------------------------------
    # 1. FECHA_NACIMIENTO siempre es yyyy-mm-dd
    df['FECHA_NACIMIENTO_DT'] = pd.to_datetime(df['FECHA_NACIMIENTO'], format='%Y-%m-%d', errors='coerce')
    
    # 2. FECHA_INGRESO y FECHAALTA cambian en 2023
    if año == '2023':
        formato_operativo = '%d-%m-%Y'
    else:
        formato_operativo = '%Y-%m-%d'
        
    df['FECHA_INGRESO_DT'] = pd.to_datetime(df['FECHA_INGRESO'], format=formato_operativo, errors='coerce')
    df['FECHAALTA_DT'] = pd.to_datetime(df['FECHAALTA'], format=formato_operativo, errors='coerce')
    
    # Calcular EDAD y DIAS_ESTADIA
    df['EDAD'] = np.floor((df['FECHA_INGRESO_DT'] - df['FECHA_NACIMIENTO_DT']).dt.days / 365.25)
    df['DIAS_ESTADIA'] = (df['FECHAALTA_DT'] - df['FECHA_INGRESO_DT']).dt.days
    
    # === LIMPIEZA FINAL DE ANOMALÍAS TEMPORALES ===
    n_estadia_neg = (df['DIAS_ESTADIA'] < 0).sum()
    n_edad_neg = (df['EDAD'] < 0).sum()
    
    if n_estadia_neg > 0 or n_edad_neg > 0:
        print(f"  [!] Limpiando anomalías reales en {año}: {n_estadia_neg} estadías < 0, {n_edad_neg} edades < 0.")
        
    # Convertimos los valores negativos (errores de digitación reales) a NaN
    df.loc[df['DIAS_ESTADIA'] < 0, 'DIAS_ESTADIA'] = np.nan
    df.loc[df['EDAD'] < 0, 'EDAD'] = np.nan

    # Borramos columnas temporales
    df.drop(columns=['FECHA_NACIMIENTO_DT', 'FECHA_INGRESO_DT', 'FECHAALTA_DT'], inplace=True)
    df.drop(columns=['FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA'], inplace=True)
    
    # ---------------------------------------------------------
    # Guardar nuevo dataset (SIN eliminar ninguna columna)
    # ---------------------------------------------------------
    nuevo_nombre = f"GRD_PROCESADO_{año}_DERIVADAS.csv"
    ruta_destino = os.path.join(carpeta_procesados, nuevo_nombre)
    df.to_csv(ruta_destino, index=False, encoding="utf-8-sig")
    
    print(f"[{año}] Guardado: {nuevo_nombre}")

print("="*60 + "\n¡Pipeline de variables derivadas completado con éxito!")

Iniciando creación de variables derivadas...
[2019] Guardado: GRD_PROCESADO_2019_DERIVADAS.csv
  [!] Limpiando anomalías reales en 2020: 10 estadías < 0, 0 edades < 0.
[2020] Guardado: GRD_PROCESADO_2020_DERIVADAS.csv
[2021] Guardado: GRD_PROCESADO_2021_DERIVADAS.csv
  [!] Limpiando anomalías reales en 2022: 1 estadías < 0, 0 edades < 0.
[2022] Guardado: GRD_PROCESADO_2022_DERIVADAS.csv
[2023] Guardado: GRD_PROCESADO_2023_DERIVADAS.csv
[2024] Guardado: GRD_PROCESADO_2024_DERIVADAS.csv
¡Pipeline de variables derivadas completado con éxito!


### Procesamiento

In [2]:
import pandas as pd
import numpy as np
import os
import re
from IPython.display import display, HTML

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_derivadas = [f"GRD_PROCESADO_{año}_DERIVADAS.csv" for año in range(2019, 2025)]

vars_cat = ['COMORBILIDAD_PRINCIPAL', 'ES_QUIRURGICO']
vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'NUM_COMORBILIDADES', 
            'NUM_PROCEDIMIENTOS', 'EDAD', 'DIAS_ESTADIA']

resultados_cat = {var: [] for var in vars_cat}
resultados_num = []

print("Leyendo datasets y calculando estadísticas (Una sola pasada)... por favor espera.\n")

for archivo in archivos_derivadas:
    ruta = os.path.join(carpeta_procesados, archivo)
    if not os.path.exists(ruta):
        continue
        
    año = re.search(r'\d{4}', archivo).group()
    df = pd.read_csv(ruta, low_memory=False)
    
    # --- 1. CATEGÓRICAS ---
    for var in vars_cat:
        if var in df.columns:
            # Calcular conteos absolutos y porcentajes
            conteos = df[var].value_counts(dropna=False)
            porcentajes = df[var].value_counts(dropna=False, normalize=True) * 100
            
            # Formatear el texto combinando ambos "N (P%)"
            freq = pd.DataFrame({
                var: conteos.index,
                'Valor_Formateado': conteos.astype(str) + " (" + porcentajes.round(2).astype(str) + "%)"
            })
            freq['Año'] = año
            resultados_cat[var].append(freq)
            
    # --- 2. NUMÉRICAS ---
    for var in vars_num:
        if var in df.columns:
            serie = df[var].dropna()
            n_total = len(serie)
            if n_total == 0: continue
                
            Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
            IQR = Q3 - Q1
            lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            
            outliers_mask = (serie < lim_inf) | (serie > lim_sup)
            n_outliers = outliers_mask.sum()
            
            resultados_num.append({
                'Año': año, 'Variable': var, 'N_Válidos': n_total,
                'Mín': serie.min(), 'Máx': serie.max(),
                'Media': round(serie.mean(), 2), 'Desv_Est': round(serie.std(), 2),
                'Outliers (N)': n_outliers, 'Outliers (%)': (n_outliers / n_total * 100).round(2)
            })

# Consolidar los resultados en DataFrames globales para consultarlos después
df_num_final = pd.DataFrame(resultados_num) if resultados_num else pd.DataFrame()
dicc_cat_final = {}

for var in vars_cat:
    if resultados_cat[var]:
        df_cat = pd.concat(resultados_cat[var], ignore_index=True)
        # Al pivotear, si un año no tiene un valor, se rellena con "0 (0.0%)"
        dicc_cat_final[var] = df_cat.pivot(index=var, columns='Año', values='Valor_Formateado').fillna("0 (0.0%)")

print("¡Cálculos terminados!")

Leyendo datasets y calculando estadísticas (Una sola pasada)... por favor espera.

¡Cálculos terminados!


In [3]:
print("--- COMORBILIDAD PRINCIPAL ---")
display(dicc_cat_final['COMORBILIDAD_PRINCIPAL'])

--- COMORBILIDAD PRINCIPAL ---


Año,2019,2020,2021,2022,2023,2024
COMORBILIDAD_PRINCIPAL,,,,,,
CODIGOS PROVISIONALES (COVID),49 (0.0%),1386 (0.18%),1748 (0.22%),1608 (0.17%),584 (0.06%),371 (0.03%)
ENDOCRINAS Y METABOLICAS,8390 (0.75%),5391 (0.7%),6470 (0.8%),7555 (0.82%),8911 (0.86%),9821 (0.91%)
INFECCIOSAS Y PARASITARIAS,2260 (0.2%),1754 (0.23%),1757 (0.22%),1971 (0.21%),2418 (0.23%),2673 (0.25%)
MATERNO-INFANTILES Y CONGENITAS,571 (0.05%),475 (0.06%),377 (0.05%),516 (0.06%),579 (0.06%),575 (0.05%)
NO_APLICA,1016813 (91.42%),705708 (91.67%),741678 (91.63%),849421 (91.64%),940799 (91.26%),978090 (90.87%)
OJO Y OIDO,1064 (0.1%),540 (0.07%),655 (0.08%),1033 (0.11%),1292 (0.13%),1323 (0.12%)
PIEL Y TEJIDO SUBCUTANEO,905 (0.08%),644 (0.08%),659 (0.08%),714 (0.08%),878 (0.09%),1092 (0.1%)
SANGRE E INMUNIDAD,5889 (0.53%),4448 (0.58%),4262 (0.53%),4705 (0.51%),5558 (0.54%),6326 (0.59%)
SINTOMAS Y HALLAZGOS,2864 (0.26%),2332 (0.3%),2543 (0.31%),2788 (0.3%),3261 (0.32%),3690 (0.34%)


In [4]:
print("--- ES QUIRÚRGICO (1=Sí, 0=No) ---")
display(dicc_cat_final['ES_QUIRURGICO'])

--- ES QUIRÚRGICO (1=Sí, 0=No) ---


Año,2019,2020,2021,2022,2023,2024
ES_QUIRURGICO,,,,,,
0,447885 (40.27%),315773 (41.02%),337116 (41.65%),348660 (37.61%),386492 (37.49%),412551 (38.33%)
1,664419 (59.73%),454042 (58.98%),472325 (58.35%),578271 (62.39%),644371 (62.51%),663847 (61.67%)


In [5]:
display(df_num_final)

,Año,Variable,N_Válidos,Mín,Máx,Media,Desv_Est,Outliers (N),Outliers (%)
0,2019,CANTIDAD_TRASLADOS,1112304,0.0,9.0,0.22,0.62,163802,14.73
1,2019,CARGA_ONCOLOGICA,1112304,0.0,8.0,0.15,0.45,135143,12.15
2,2019,NUM_COMORBILIDADES,1112304,0.0,35.0,3.19,2.76,53931,4.85
3,2019,NUM_PROCEDIMIENTOS,1112304,1.0,30.0,6.82,5.24,42307,3.80
4,2019,EDAD,1112304,0.0,124.0,43.09,25.83,0,0.00
5,2019,DIAS_ESTADIA,1112304,0.0,2322.0,5.30,12.85,121103,10.89
6,2020,CANTIDAD_TRASLADOS,769815,0.0,9.0,0.32,0.75,162427,21.10
7,2020,CARGA_ONCOLOGICA,769815,0.0,9.0,0.15,0.45,90034,11.70
8,2020,NUM_COMORBILIDADES,769815,0.0,35.0,3.89,3.38,28228,3.67
9,2020,NUM_PROCEDIMIENTOS,769815,1.0,30.0,8.40,5.97,33966,4.41


In [8]:
import pandas as pd
import os
from IPython.display import display

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_derivadas = [f"GRD_PROCESADO_{año}_DERIVADAS.csv" for año in range(2019, 2025)]

vars_num = [
    'CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'NUM_COMORBILIDADES', 
    'NUM_PROCEDIMIENTOS', 'EDAD', 'DIAS_ESTADIA'
]

print("Leyendo y calculando nulos... por favor espera.\n")

resultados_nulos = []
totales_anuales = {}

for archivo in archivos_derivadas:
    ruta = os.path.join(carpeta_procesados, archivo)
    if not os.path.exists(ruta):
        continue
        
    año = archivo[-17:-13]
    df = pd.read_csv(ruta, usecols=vars_num, low_memory=False)
    
    total_filas = len(df)
    totales_anuales[año] = total_filas
    nulos = df.isna().sum()
    
    for var in vars_num:
        cant_nulos = nulos[var]
        pct_nulos = (cant_nulos / total_filas) * 100
        
        # Formatear como "N (P%)"
        texto_nulos = f"{cant_nulos} ({pct_nulos:.3f}%)" if cant_nulos > 0 else "0"
        
        resultados_nulos.append({
            'Año': año,
            'Variable': var,
            'Nulos_Formateados': texto_nulos
        })

# Crear DataFrame y pivotear
if resultados_nulos:
    df_nulos = pd.DataFrame(resultados_nulos)
    tabla_final = df_nulos.pivot(index='Variable', columns='Año', values='Nulos_Formateados')
    
    # Imprimir un resumen de los totales de egresos primero
    print("=== TOTAL DE EGRESOS POR AÑO ===")
    for anio, total in totales_anuales.items():
        print(f"{anio}: {total:,}".replace(',', '.'))
        
    print("\n=== RECUENTO DE NULOS (NaN) EN VARIABLES DERIVADAS ===")
    display(tabla_final)
else:
    print("No se encontraron archivos procesados para analizar.")

Leyendo y calculando nulos... por favor espera.

=== TOTAL DE EGRESOS POR AÑO ===
019_: 1.112.304
020_: 769.815
021_: 809.441
022_: 926.931
023_: 1.030.863
024_: 1.076.398

=== RECUENTO DE NULOS (NaN) EN VARIABLES DERIVADAS ===


Año,019_,020_,021_,022_,023_,024_
Variable,,,,,,
CANTIDAD_TRASLADOS,0,0,0,0,0,0
CARGA_ONCOLOGICA,0,0,0,0,0,0
DIAS_ESTADIA,0,10 (0.001%),0,1 (0.000%),0,50 (0.005%)
EDAD,0,1 (0.000%),0,7 (0.001%),10 (0.001%),53 (0.005%)
NUM_COMORBILIDADES,0,0,0,0,0,0
NUM_PROCEDIMIENTOS,0,0,0,0,0,0


In [9]:
import pandas as pd
import os
import re

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_derivadas = [f"GRD_PROCESADO_{año}_DERIVADAS.csv" for año in range(2019, 2025)]

print("Iniciando eliminación de valores nulos residuales...\n" + "="*60)

# Variables donde no permitiremos ningún NaN
columnas_criticas = ['EDAD', 'DIAS_ESTADIA']

for archivo in archivos_derivadas:
    ruta = os.path.join(carpeta_procesados, archivo)
    if not os.path.exists(ruta):
        continue
        
    # Extracción segura del año usando Regex
    año = re.search(r'\d{4}', archivo).group()
    
    # Leer el dataset
    df = pd.read_csv(ruta, low_memory=False)
    filas_iniciales = len(df)
    
    # Eliminar filas que tengan NaN en las columnas críticas
    df.dropna(subset=columnas_criticas, inplace=True)
    
    filas_finales = len(df)
    eliminados = filas_iniciales - filas_finales
    
    # Guardar el dataset limpio sobreescribiendo el archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    
    print(f"[{año}] Pacientes iniciales: {filas_iniciales:,}".replace(',', '.'))
    print(f"       Pacientes eliminados: {eliminados} filas con NaN")
    print(f"       Pacientes finales: {filas_finales:,} (Guardado)\n".replace(',', '.'))

print("="*60 + "\n¡Limpieza completada! El dataset está 100% libre de nulos.")

Iniciando eliminación de valores nulos residuales...
[2019] Pacientes iniciales: 1.112.304
       Pacientes eliminados: 0 filas con NaN
       Pacientes finales: 1.112.304 (Guardado)

[2020] Pacientes iniciales: 769.815
       Pacientes eliminados: 11 filas con NaN
       Pacientes finales: 769.804 (Guardado)

[2021] Pacientes iniciales: 809.441
       Pacientes eliminados: 0 filas con NaN
       Pacientes finales: 809.441 (Guardado)

[2022] Pacientes iniciales: 926.931
       Pacientes eliminados: 8 filas con NaN
       Pacientes finales: 926.923 (Guardado)

[2023] Pacientes iniciales: 1.030.863
       Pacientes eliminados: 10 filas con NaN
       Pacientes finales: 1.030.853 (Guardado)

[2024] Pacientes iniciales: 1.076.398
       Pacientes eliminados: 53 filas con NaN
       Pacientes finales: 1.076.345 (Guardado)

¡Limpieza completada! El dataset está 100% libre de nulos.
